# 🧬 ClimateEnzyme Discovery Pipeline

## AI-Guided Discovery of PET-Degrading Enzymes

**Run entirely in your browser - no installation needed!**

This notebook:
1. Mines enzyme sequences from UniProt
2. Predicts structures with AlphaFold2 (ColabFold)
3. Analyzes active sites and functional features
4. Ranks candidates for experimental testing

---

**Time needed:** ~30-60 minutes depending on number of sequences

**Requirements:** Just a Google account!

---

In [ ]:
#@title 1️⃣ Setup - Install Dependencies (Run this first!)
#@markdown This installs everything needed. Takes ~2 minutes.

%%capture
!pip install requests

# Clone the pipeline repository
!git clone https://github.com/adamrnardis-collab/Polytrad.git 2>/dev/null || (cd Polytrad && git pull)

import sys
sys.path.insert(0, '/content/Polytrad/src')

print("✅ Setup complete!")

In [ ]:
#@title 2️⃣ Mine Enzyme Sequences from UniProt
#@markdown This searches UniProt for PET-degrading enzymes.

max_sequences = 50  #@param {type:"slider", min:10, max:200, step:10}
include_cutinases = True  #@param {type:"boolean"}
include_petases = True  #@param {type:"boolean"}

from sequence_mining import SequenceMiner, FilterConfig
import os

os.makedirs('/content/data/sequences', exist_ok=True)

config = FilterConfig(
    min_length=200,
    max_length=500,
    max_sequences=max_sequences
)

miner = SequenceMiner('/content/data/sequences')

enzyme_types = []
if include_petases:
    enzyme_types.extend(['petase', 'mhetase'])
if include_cutinases:
    enzyme_types.extend(['cutinase', 'thermostable_cutinase'])

print("🔍 Searching UniProt for enzyme sequences...\n")
sequences = miner.mine_pet_enzymes(enzyme_types=enzyme_types, config=config)

print("\n📥 Adding reference enzymes (IsPETase, LCC, TfCut2)...")
miner.add_reference_sequences()

fasta_path = miner.save_fasta('candidates.fasta')
miner.save_metadata('metadata.tsv')

print(f"\n✅ Found {len(miner.sequences)} enzyme candidates!")
print(f"📁 Saved to: {fasta_path}")

# Show summary
print("\n" + "="*50)
print("TOP 10 CANDIDATES")
print("="*50)
for i, seq in enumerate(miner.sequences[:10]):
    reviewed = "✓" if seq.reviewed else " "
    print(f"{i+1:2}. [{reviewed}] {seq.accession:<12} {seq.organism[:30]:<30} {seq.length} aa")
print("="*50)

In [ ]:
#@title 3️⃣ View Mined Sequences
#@markdown Shows the sequences that were found.

import pandas as pd

# Load metadata
df = pd.read_csv('/content/data/sequences/metadata.tsv', sep='\t')

print(f"Total sequences: {len(df)}")
print(f"Reviewed (Swiss-Prot): {df['reviewed'].sum()}")
print(f"Unique organisms: {df['organism'].nunique()}")
print(f"Length range: {df['length'].min()}-{df['length'].max()} aa")
print()

# Display table
df[['accession', 'organism', 'length', 'ec_number', 'reviewed']].head(20)

---
## 🔬 Structure Prediction with AlphaFold2

This section uses ColabFold to predict 3D structures.

**⚠️ Important:** Enable GPU first!
- Go to **Runtime → Change runtime type → GPU**
- This is FREE on Colab!

---

In [ ]:
#@title 4️⃣ Install ColabFold (AlphaFold2)
#@markdown This installs the structure prediction software. Takes ~3 minutes.

%%capture
!pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"

import torch
if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
#@title 5️⃣ Run Structure Predictions
#@markdown Predicts 3D structures for all sequences. Takes ~5-15 min per protein.

#@markdown **Tip:** Start with fewer sequences (10-20) for testing.

num_sequences = 10  #@param {type:"slider", min:1, max:50, step:1}
num_models = 1  #@param {type:"slider", min:1, max:5, step:1}
use_amber = False  #@param {type:"boolean"}

import os
os.makedirs('/content/data/structures', exist_ok=True)

# Create subset FASTA with selected number of sequences
from utils import FastaHandler

all_seqs = FastaHandler.read('/content/data/sequences/candidates.fasta')
subset = all_seqs[:num_sequences]

subset_file = '/content/data/sequences/subset.fasta'
FastaHandler.write(subset, subset_file)

print(f"🧬 Predicting structures for {len(subset)} sequences...")
print(f"⏱️ Estimated time: {len(subset) * 5}-{len(subset) * 15} minutes\n")

from colabfold.batch import run as run_colabfold
from colabfold.download import download_alphafold_params

print("📥 Downloading AlphaFold2 parameters...")
download_alphafold_params("alphafold2_ptm")

print("\n🔮 Starting predictions...\n")
run_colabfold(
    subset_file,
    '/content/data/structures',
    num_models=num_models,
    num_recycle=3,
    use_amber=use_amber,
    use_templates=False,
    msa_mode="mmseqs2_uniref_env",
    model_type="alphafold2_ptm"
)

print("\n✅ Structure predictions complete!")

In [ ]:
#@title 6️⃣ Parse Prediction Results
#@markdown Extracts quality scores from predictions.

from structure_prediction import StructureParser

parser = StructureParser('/content/data/structures')
predictions = parser.parse_results_directory(verbose=True)

if predictions:
    parser.save_summary()
    
    print("\n" + "="*60)
    print("STRUCTURE PREDICTION SUMMARY")
    print("="*60)
    print(f"{'ID':<30} {'pLDDT':>8} {'Category':<15}")
    print("-"*60)
    
    for p in sorted(predictions, key=lambda x: x.plddt_mean, reverse=True):
        stars = "***" if p.plddt_mean >= 90 else "**" if p.plddt_mean >= 70 else "*"
        print(f"{p.sequence_id[:30]:<30} {p.plddt_mean:>7.1f}{stars} {p.confidence_category:<15}")
    print("="*60)
else:
    print("⚠️ No predictions found. Run step 5 first.")

In [ ]:
#@title 7️⃣ Analyze Structures
#@markdown Detects active sites and catalytic features.

from structure_analysis import analyze_all_structures
import os

os.makedirs('/content/data/analysis', exist_ok=True)

print("🔬 Analyzing predicted structures...\n")

results = analyze_all_structures(
    structures_dir='/content/data/structures',
    output_file='/content/data/analysis/structure_analysis.json',
    verbose=True
)

print(f"\n✅ Analyzed {len(results)} structures!")

In [ ]:
#@title 8️⃣ Rank Candidates
#@markdown Creates final rankings based on all criteria.

from ranking import run_ranking

print("📊 Calculating rankings...\n")

ranked = run_ranking(
    sequences_file='/content/data/sequences/metadata.tsv',
    structures_file='/content/data/structures/structure_summary.csv',
    analysis_file='/content/data/analysis/structure_analysis.json',
    output_dir='/content/results',
    verbose=True
)

print("\n" + "🏆"*20)
print("\n🏆 TOP ENZYME CANDIDATES FOR PET DEGRADATION 🏆\n")
print("🏆"*20 + "\n")

for c in ranked[:10]:
    medal = "🥇" if c.rank == 1 else "🥈" if c.rank == 2 else "🥉" if c.rank == 3 else "  "
    print(f"{medal} #{c.rank}: {c.sequence_id}")
    print(f"   Organism: {c.organism or 'Unknown'}")
    print(f"   Score: {c.overall_score:.3f} | pLDDT: {c.plddt_mean:.1f}")
    print(f"   {c.functional_notes[:80]}")
    print()

In [ ]:
#@title 9️⃣ View Results Report
#@markdown Displays the HTML report in the notebook.

from IPython.display import HTML, display

with open('/content/results/ranking_report.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))

In [ ]:
#@title 🔟 Download All Results
#@markdown Downloads everything as a ZIP file.

import shutil
from google.colab import files

# Create zip archive
shutil.make_archive('/content/ClimateEnzyme_Results', 'zip', '/content/results')

print("📦 Created ClimateEnzyme_Results.zip")
print("\nContents:")
print("  - ranked_candidates.csv (spreadsheet)")
print("  - ranked_candidates.json (data)")
print("  - ranking_report.html (visual report)")
print("  - ranking_report.md (markdown)")
print("\n⬇️ Downloading...")

files.download('/content/ClimateEnzyme_Results.zip')

---

## 🎉 Pipeline Complete!

### What you now have:

1. **Enzyme sequences** from UniProt database
2. **Predicted 3D structures** from AlphaFold2
3. **Active site analysis** for each candidate
4. **Ranked list** of best candidates for testing

### Next steps:

1. Review the top candidates in the HTML report
2. Send promising sequences to a lab for synthesis
3. Test for PET degradation activity

### Learn more:

- [GitHub Repository](https://github.com/adamrnardis-collab/Polytrad)
- [Why This Matters](https://github.com/adamrnardis-collab/Polytrad/blob/main/docs/WHY_THIS_MATTERS.md)

---

*Helping enzymes help the planet* 🌍
